# 🔍 Exercises XP: Vector Databases and RAG

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation
- Vector database utility (similarity search, RAG)
- Differences between vector DBs, libraries, and plugins
- Best practices for vector store usage and performance
- How LMs use context; embedding generation and storage
- Querying vector stores and applying LMs for QA with retrieved context

## What you'll build
A functional RAG pipeline with **FAISS** and **ChromaDB**, plus QA over retrieved context using a Hugging Face model.

## Pipeline overview
```
CSV Data  →  Sentence Embeddings  →  FAISS Index  →  Similarity Search
                                  →  ChromaDB     →  QA with Flan-T5
```

---
## 0. Setup — Install dependencies

In [ ]:
# Run once — restart runtime after if prompted
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

# System dependency for FAISS on Linux (Colab / Ubuntu)
import subprocess
subprocess.run(["apt", "install", "-y", "libomp-dev"], capture_output=True)

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from IPython.display import display

# Create cache directory for intermediate files
os.makedirs('cache', exist_ok=True)

print("✅ All imports successful")
print(f"   numpy  : {np.__version__}")
print(f"   faiss  : {faiss.__version__}")
print(f"   pandas : {pd.__version__}")

---
## 🌟 Exercise 1 — Data loading and preparation

**Why this matters:** Before diving into vector search, we need clean, well-structured data with unique identifiers. A manageable subset speeds up iteration during development.

> Dataset: `labelled_newscatcher_dataset.csv` — labelled news articles with title, topic, and other metadata.

In [ ]:
# ── 1. Load the dataset ───────────────────────────────────────────────
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')

print(f"Dataset shape : {pdf.shape}")
print(f"Columns       : {pdf.columns.tolist()}")
print(f"Missing values:\n{pdf.isnull().sum()}")

# ── 2. Add a unique identifier column (if not already present) ────────
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))   # simple 0-based integer ID
    print("\n✅ 'id' column added")
else:
    print("\n✅ 'id' column already present")

# ── 3. Inspect the data ───────────────────────────────────────────────
display(pdf.head())
print(f"\nTopic distribution:\n{pdf['topic'].value_counts()}")

In [ ]:
# ── 4. Create a manageable subset (first 1 000 rows) ─────────────────
# Using a subset speeds up embedding generation and indexing
SUBSET_SIZE = 1000
pdf_subset = pdf.head(SUBSET_SIZE).copy().reset_index(drop=True)

print(f"Subset shape : {pdf_subset.shape}")
display(pdf_subset[['id', 'title', 'topic']].head(10))

---
## 🌟 Exercise 2 — Vectorization with Sentence Transformers

**Why this matters:** Machines cannot process raw text directly. We convert each article title into a **dense embedding vector** that captures its semantic meaning, enabling similarity comparisons.

Model: `all-MiniLM-L6-v2` — fast, lightweight, high-quality embeddings (384 dimensions).

In [ ]:
# ── Helper function: format rows as InputExample objects ──────────────
def example_create_fn(idx: int, text: str) -> InputExample:
    """
    Convert a (index, text) pair into a SentenceTransformer InputExample.

    Args:
        idx  : unique integer identifier for this example
        text : the text to embed (article title)

    Returns:
        InputExample with guid=str(idx), texts=[text], label=0.0
    """
    return InputExample(guid=str(idx), texts=[text], label=0.0)


# ── Create training examples from the subset ──────────────────────────
faiss_train_examples = [
    example_create_fn(row['id'], row['title'])
    for _, row in pdf_subset.iterrows()
]

print(f"✅ Created {len(faiss_train_examples)} InputExample objects")
print("\nFirst 2 examples:")
for ex in faiss_train_examples[:2]:
    print(f"  guid={ex.guid}  text='{ex.texts[0][:60]}...'")

In [ ]:
# ── Initialize the Sentence Transformer model ─────────────────────────
# all-MiniLM-L6-v2: 22M params, 384-dim vectors, ~5x faster than BERT-base
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model max sequence length : {model.max_seq_length}")

# ── Extract titles as a plain list of strings ─────────────────────────
titles_list = pdf_subset['title'].tolist()
print(f"Titles to embed           : {len(titles_list)}")
print(f"Sample title              : '{titles_list[0]}'")

# ── Generate embeddings ───────────────────────────────────────────────
# convert_to_numpy=True returns a (N, 384) numpy array
faiss_title_embedding = model.encode(
    titles_list,
    convert_to_numpy=True,
    show_progress_bar=True
)

n_embeddings, embed_dim = len(faiss_title_embedding), len(faiss_title_embedding[0])
print(f"\n✅ Embeddings generated")
print(f"   Shape : ({n_embeddings}, {embed_dim})")
print(f"   → {n_embeddings} vectors of {embed_dim} dimensions each")
print(f"   Sample (first 8 dims): {faiss_title_embedding[0][:8].round(4)}")

---
## 🌟 Exercise 3 — FAISS indexing and search

**Why this matters:** FAISS (Facebook AI Similarity Search) enables **millisecond-scale similarity search** over millions of vectors. We use `IndexFlatIP` (inner product = cosine similarity for normalized vectors) wrapped in `IndexIDMap` to keep track of original article IDs.

Search flow: `query string → embed → normalize → FAISS.search → retrieve articles`

In [ ]:
# ── Prepare arrays for indexing ───────────────────────────────────────
pdf_to_index = pdf_subset.copy()

# ID array must be int64 for FAISS IndexIDMap
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)

# Embeddings must be float32 for FAISS
content_encoded_normalized = faiss_title_embedding.astype('float32')

# ── Normalize vectors (required for cosine similarity) ────────────────
# After L2 normalization: ||v|| = 1  →  inner product = cosine similarity
faiss.normalize_L2(content_encoded_normalized)

# ── Build FAISS index ─────────────────────────────────────────────────
# IndexFlatIP  : exact inner-product search (no approximation)
# IndexIDMap   : wraps the index so search returns our article IDs
index_content = faiss.IndexIDMap(
    faiss.IndexFlatIP(content_encoded_normalized.shape[1])  # dim = 384
)
index_content.add_with_ids(content_encoded_normalized, id_index)

print(f"✅ FAISS index built")
print(f"   Vectors indexed : {index_content.ntotal}")
print(f"   Vector dimension: {content_encoded_normalized.shape[1]}")

In [ ]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3) -> pd.DataFrame:
    """
    Encode a query string, search the FAISS index, and return the
    k most similar articles with their similarity scores.

    Args:
        query        : natural-language search string
        pdf_to_index : DataFrame containing the indexed articles
        k            : number of nearest neighbours to retrieve

    Returns:
        DataFrame of matching articles with an added 'similarities' column
    """
    # 1. Encode the query into an embedding vector
    query_vector = model.encode(
        [query],                # must be a list
        convert_to_numpy=True
    ).astype('float32')

    # 2. Normalize the query vector (same space as indexed vectors)
    faiss.normalize_L2(query_vector)

    # 3. Search: returns (similarities, ids) arrays of shape (1, k)
    sims, ids = index_content.search(query_vector, k)

    # 4. Retrieve matching articles from the DataFrame
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]

    # Sort by similarity descending
    results = results.sort_values('similarities', ascending=False)
    return results


# ── Test the search function ──────────────────────────────────────────
print("=== Search: 'animal' ===")
display(search_content('animal', pdf_to_index, k=5)[['id', 'title', 'topic', 'similarities']])

print("\n=== Search: 'space exploration' ===")
display(search_content('space exploration', pdf_to_index, k=5)[['id', 'title', 'topic', 'similarities']])

print("\n=== Search: 'economy financial crisis' ===")
display(search_content('economy financial crisis', pdf_to_index, k=5)[['id', 'title', 'topic', 'similarities']])

---
## 🌟 Exercise 4 — ChromaDB collection and querying

**Why this matters:** Unlike FAISS (a pure similarity-search library), ChromaDB is a **full vector database** that handles tokenization, embedding, indexing, and metadata storage automatically. It's ideal for RAG pipelines that need to store, filter, and retrieve documents alongside their metadata.

| | FAISS | ChromaDB |
|--|-------|----------|
| Embedding | Manual | Automatic |
| Metadata | ❌ | ✅ |
| Persistence | Manual | Built-in |
| Best for | Speed at scale | RAG / LLM apps |

In [ ]:
# ── Initialize ChromaDB client ────────────────────────────────────────
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'

# Delete existing collection to avoid conflicts on re-run
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)
    print(f"♻️  Deleted existing collection '{collection_name}'")

# ── Create the collection ─────────────────────────────────────────────
# ChromaDB uses its default SentenceTransformerEmbeddingFunction
# (all-MiniLM-L6-v2) when no embedding_function is specified
collection = chroma_client.create_collection(name=collection_name)
print(f"✅ Collection '{collection_name}' created")

# ── Add documents (first 100 titles) ─────────────────────────────────
# documents : list of text strings to embed and store
# metadatas : list of dicts with additional info per document
# ids       : unique string identifiers (ChromaDB requires strings)
CHROMA_SUBSET = 100

collection.add(
    documents=pdf_subset['title'][:CHROMA_SUBSET].tolist(),
    metadatas=[
        {"topic": str(topic)}
        for topic in pdf_subset['topic'][:CHROMA_SUBSET].tolist()
    ],
    ids=[
        str(i)
        for i in pdf_subset['id'][:CHROMA_SUBSET].tolist()
    ]
)

print(f"✅ {CHROMA_SUBSET} documents added to the collection")
print(f"   Collection count: {collection.count()}")

In [ ]:
# ── Query the collection ──────────────────────────────────────────────
# ChromaDB automatically embeds the query_texts and returns nearest neighbours
results = collection.query(
    query_texts=["space"],   # the search term
    n_results=10             # retrieve top-10 most relevant documents
)

print("=== ChromaDB Query Results: 'space' (top 10) ===")
print(json.dumps(results, indent=2))

# ── Pretty summary ────────────────────────────────────────────────────
print("\n--- Formatted Results ---")
for i, (doc, meta, dist) in enumerate(
    zip(results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]), 1
):
    print(f"  [{i}] distance={dist:.4f}  topic={meta['topic']:<15}  '{doc[:70]}'")

---
## 🌟 Exercise 5 — Question Answering with a Hugging Face model (RAG)

**Why this matters:** Retrieval alone isn't enough — we need a language model to **synthesize** the retrieved documents into a coherent answer. This is the core of **RAG** (Retrieval-Augmented Generation):
1. **Retrieve** relevant documents from ChromaDB
2. **Augment** the prompt with the retrieved context
3. **Generate** an answer using a language model

We use `google/flan-t5-small` — a lightweight instruction-tuned model that outperforms GPT-2 on QA tasks.

In [ ]:
# ── Load the model and tokenizer ──────────────────────────────────────
# Flan-T5 is a seq2seq model (encoder-decoder), not causal LM
# → use AutoModelForSeq2SeqLM and "text2text-generation" pipeline
model_id = 'google/flan-t5-small'

print(f"Loading model: {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model  = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# ── Build the text2text pipeline ──────────────────────────────────────
pipe = pipeline(
    "text2text-generation",   # Flan-T5 is seq2seq, not causal
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    device_map="auto",        # GPU if available, else CPU
)

print(f"✅ Pipeline ready: {model_id}")

In [ ]:
# ── Define the user question ──────────────────────────────────────────
question = "What's the latest news on space development?"

# ── Retrieve relevant context from ChromaDB ───────────────────────────
# Reuse the results from Exercise 4 (query: "space")
context_docs = results['documents'][0][:3]   # top-3 documents
context = ' '.join(context_docs)

print("=== Retrieved Context ===")
for i, doc in enumerate(context_docs, 1):
    print(f"  [{i}] {doc}")

# ── Build the RAG prompt ──────────────────────────────────────────────
# Flan-T5 performs best with explicit instruction-following prompts
prompt = (
    f"Answer the question using only the context below.\n"
    f"Context: {context}\n"
    f"Question: {question}\n"
    f"Answer:"
)

print("\n=== Prompt sent to the model ===")
print(prompt)

In [ ]:
# ── Generate the answer ───────────────────────────────────────────────
lm_response = pipe(prompt)
answer = lm_response[0]['generated_text']

print("=" * 60)
print("  RAG ANSWER")
print("=" * 60)
print(f"Question : {question}")
print(f"Answer   : {answer}")

In [ ]:
# ── Experiment: vary query, context window size, and question ─────────

def rag_qa(question: str, search_term: str, n_context_docs: int = 3) -> str:
    """
    Full RAG pipeline:
      1. Retrieve n_context_docs from ChromaDB using search_term
      2. Build an instruction prompt
      3. Generate an answer with Flan-T5

    Args:
        question       : natural-language question to answer
        search_term    : term used to retrieve relevant context
        n_context_docs : number of documents to include in the prompt

    Returns:
        Generated answer string
    """
    # Retrieve
    retrieved = collection.query(
        query_texts=[search_term],
        n_results=n_context_docs
    )
    docs    = retrieved['documents'][0]
    context = ' '.join(docs)

    # Build prompt
    prompt = (
        f"Answer the question using only the context below.\n"
        f"Context: {context}\n"
        f"Question: {question}\n"
        f"Answer:"
    )

    # Generate
    response = pipe(prompt)[0]['generated_text']
    return response


# ── Run experiments ───────────────────────────────────────────────────
experiments = [
    ("What's the latest news on space development?",  "space",    3),
    ("What is happening in the world of sports?",      "sports",   5),
    ("What are scientists discovering about health?",  "health",   3),
    ("What is the current state of the economy?",      "economy",  5),
]

print("=" * 70)
print("  RAG EXPERIMENTS")
print("=" * 70)
for question, term, n_docs in experiments:
    answer = rag_qa(question, term, n_docs)
    print(f"\n📌 Q  : {question}")
    print(f"   🔍 Search term  : '{term}' | Context docs: {n_docs}")
    print(f"   💬 Answer       : {answer}")
    print("-" * 70)

---
## 📋 Summary

| Exercise | Component | Purpose |
|----------|-----------|--------|
| 1 | Pandas + CSV | Load & subset 1 000 news articles |
| 2 | `all-MiniLM-L6-v2` | Generate 384-dim semantic embeddings |
| 3 | FAISS `IndexFlatIP` | Fast cosine similarity search |
| 4 | ChromaDB | Full vector DB with metadata & auto-embedding |
| 5 | Flan-T5 + RAG | QA over retrieved context |

### Key takeaways

**FAISS vs ChromaDB:**
- FAISS is a **library** — you manage embeddings, IDs, and metadata yourself. Best for raw speed at scale.
- ChromaDB is a **database** — it handles embedding, storage, and metadata automatically. Best for RAG / LLM apps.

**RAG benefits:**
- LMs like Flan-T5 have a fixed knowledge cutoff. RAG gives them access to **real-time, domain-specific context** without fine-tuning.
- The quality of retrieval directly impacts answer quality — better embeddings → better answers.

**Next steps:**
1. Use ANN indexes (`IndexIVFFlat`, `IndexHNSW`) for sub-linear search on large corpora
2. Add metadata filters in ChromaDB (e.g., filter by `topic`)
3. Upgrade to a larger model (`flan-t5-large`, `mistral-7b`) for richer answers
4. Evaluate retrieval quality with Recall@K and NDCG metrics